# M3L3 E09 - Delegacion paralela con LangGraph
### Modulo 3 - Lecture 3 - Sistemas Multiagente

**Ejercicio paralelo:** E04 (consulta mixta y delegacion paralela)

## Que vas a aprender hoy

- Detectar multiples intenciones en una sola consulta.
- Crear fan-out dinamico con la `Send` API de LangGraph.
- Usar `Annotated[list, operator.add]` para acumular resultados de ramas paralelas.
- Entender la diferencia entre routing condicional simple y delegacion paralela.

Este ejercicio responde una pregunta clave: que pasa si una consulta pertenece a mas de un dominio al mismo tiempo.

## Que necesitas saber antes

En E08 hiciste routing condicional: una consulta entraba, el router elegia un destino, y solo se ejecutaba una rama.

En E09 subimos un nivel: una consulta puede necesitar varios especialistas al mismo tiempo. Por ejemplo:

> Tengo un problema con la VPN y necesito pedir vacaciones.

Esa consulta tiene Tech y HR. No alcanza con elegir un solo agente.

Conceptos:

- **Fan-out:** un nodo dispara varias ramas.
- **Fan-in:** varias ramas vuelven a un punto comun.
- **Send:** objeto de LangGraph que crea una rama dinamica.
- **State acumulador:** campo del estado preparado para recibir resultados de varias ramas.

| E08 router simple | E09 fan-out paralelo |
|---|---|
| Una consulta -> un destino | Una consulta -> varios destinos |
| Devuelve string de ruta | Devuelve lista de `Send` |
| Un agente responde | Varios especialistas responden |
| No necesita merge complejo | Necesita acumular resultados |

## Paso 1 - Elegir proveedor de LLM

El LLM se usa en dos lugares:

1. Para detectar que dominios aparecen en la consulta.
2. Para que cada especialista redacte su respuesta usando su contexto.

Usamos una variable global `llm` para que el resto del notebook no dependa del proveedor elegido.

In [ ]:
# Elegimos proveedor. Para la clase, OpenAI es el camino recomendado.
# Si cambias PROVIDER, se instala solo el paquete necesario para ese proveedor.
PROVIDER = "openai"   # opciones: "openai" | "gemini" | "claude"

# os.environ guarda la API key solo durante la sesion del notebook.
# getpass oculta la key en la salida para no exponer secretos.
import os
from getpass import getpass

if PROVIDER == "openai":
    # Wrapper de LangChain para modelos de OpenAI.
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ").strip()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    # Misma interfaz de chat, distinto proveedor por debajo.
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ").strip()
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    # Claude tambien queda expuesto como objeto `llm` para que el resto del notebook no cambie.
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ").strip()
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER invalido: {PROVIDER!r}. Opciones: openai | gemini | claude")

print(f"LLM listo -> proveedor: {PROVIDER}")

In [ ]:
# LangGraph nos da StateGraph, START, END y Send.
# - StateGraph: contenedor del flujo.
# - START / END: nodos virtuales de inicio y fin.
# - Send: API para disparar ramas dinamicas en paralelo.
!pip install langgraph -q

# operator.add se usa con Annotated para decir: "si varias ramas escriben en results, sumalas".
import operator

# TypedDict documenta la forma del estado.
# Annotated agrega una regla de fusion sobre un campo del estado.
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

print("LangGraph listo.")

## Seccion 1 - La `Send` API y el State acumulador

`Send(nodo, estado_parcial)` le dice a LangGraph: ejecuta este nodo con este estado parcial.

Si el dispatcher devuelve tres `Send`, LangGraph crea tres ramas:

```text
START
  |
  v
dispatcher
  |-- Send("specialist", {domain="hr", query=...})
  |-- Send("specialist", {domain="tech", query=...})
  |-- Send("specialist", {domain="billing", query=...})
          |
          v
        merge
          |
          v
         END
```

El problema tecnico aparece cuando varias ramas escriben en el mismo campo `results`.

```python
# Mal para ramas paralelas: una rama puede pisar a otra.
results: list[str]

# Bien: cada rama agrega su resultado a la lista.
results: Annotated[list[str], operator.add]
```

Ese `operator.add` es la regla de fusion: cuando llegan varias listas, se concatenan.

In [ ]:
# Base de conocimiento chica para que el foco sea el fan-out, no el volumen documental.
# Cada dominio representa un agente especialista posible.
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 dias habiles por ano.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contrasena: restablecer desde el portal de identidad.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del dia 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
    ],
}


def retrieve(domain: str, query: str) -> str:
    # Retriever minimo: elige el documento del dominio con mas palabras compartidas con la consulta.
    # En produccion esto podria reemplazarse por embeddings + vector store.
    docs = knowledge_base[domain]
    words = set(query.lower().split())
    return max(docs, key=lambda d: sum(1 for w in words if w in d.lower()))


def detect_intents_llm(query: str) -> list:
    """Usa el LLM para detectar multiples dominios en la consulta."""
    # Este prompt no pide una respuesta al usuario. Pide una decision estructural para el grafo.
    prompt = (
        "Detecta todos los dominios relevantes en esta consulta de soporte corporativo.\n"
        "Dominios validos: hr, tech, billing\n"
        "Responde SOLO con los dominios separados por coma, sin espacios extras.\n"
        "Si ninguno aplica, responde exactamente: unknown\n\n"
        f"Consulta: {query}"
    )
    response = llm.invoke(prompt)

    # Normalizamos la salida para protegernos de mayusculas o espacios.
    raw = response.content.strip().lower()
    domains = [d.strip() for d in raw.split(",")]

    # Solo dejamos dominios que el grafo sabe manejar.
    valid = [d for d in domains if d in ("hr", "tech", "billing")]
    return valid if valid else ["unknown"]

print("Helpers listos.")

In [ ]:
class SpecialistInput(TypedDict):
    # Estado parcial que recibe cada rama specialist.
    # No recibe todo ParallelState: solo lo necesario para resolver su dominio.
    domain: str
    query: str


class ParallelState(TypedDict):
    # Consulta original del usuario.
    query: str

    # Campo acumulador: cada rama specialist devuelve una lista y LangGraph las concatena.
    results: Annotated[list[str], operator.add]

    # Respuesta final despues del merge.
    final: str

## Seccion 2 - Nodos del fan-out

Vamos a definir tres piezas:

- `specialist`: resuelve un dominio puntual.
- `dispatcher`: detecta dominios y crea una lista de `Send`.
- `merge`: une los resultados acumulados en una respuesta final.

En Starter, los alumnos completan `dispatcher` y `merge`. El especialista ya esta armado para que el foco sea LangGraph y no el prompt.

In [ ]:
def specialist(state: SpecialistInput) -> dict:
    # Esta funcion corre una vez por cada dominio enviado por dispatcher.
    domain = state["domain"]

    # Si dispatcher manda unknown, no hay knowledge base especializada.
    if domain not in knowledge_base:
        return {"results": ["No puedo responder esa consulta."]}

    # RAG simple: tomamos todos los documentos del dominio como contexto.
    context = "\n".join(knowledge_base[domain])

    # El LLM responde desde el contexto del especialista.
    response = llm.invoke(
        f"Usando solo este contexto:\n{context}\n\nResponde brevemente en espanol: {state['query']}"
    )

    # Devolvemos lista porque `results` es un acumulador.
    return {"results": [f"{domain.upper()}: {response.content.strip()}"]}


def dispatcher(state: ParallelState) -> list:
    # TODO 1: detectar dominios con detect_intents_llm(state["query"]).
    # TODO 2: devolver una lista de Send, uno por dominio detectado.
    # Pista:
    # return [Send("specialist", {"domain": d, "query": state["query"]}) for d in domains]
    return []


def merge(state: ParallelState) -> dict:
    # TODO: unir state["results"] en un string con saltos de linea.
    # Pista: "\n".join(state["results"])
    return {"final": ""}

## Seccion 3 - Construir el grafo de fan-out

La linea importante es:

```python
graph.add_conditional_edges("dispatcher", lambda x: x, ["specialist"])
```

Lectura:

- `dispatcher` es el nodo origen.
- `lambda x: x` devuelve directamente la lista de `Send` que produjo dispatcher.
- `["specialist"]` declara que el unico nodo destino permitido para esas ramas es `specialist`.

Despues cada rama `specialist` cae en `merge`, y `merge` termina el flujo.

In [ ]:
# Creamos el grafo con el estado global ParallelState.
graph = StateGraph(ParallelState)

# Registramos nodos.
# dispatcher decide ramas, specialist responde por dominio, merge une resultados.
graph.add_node("dispatcher", dispatcher)
graph.add_node("specialist",  specialist)
graph.add_node("merge",       merge)

# El flujo siempre empieza en dispatcher.
graph.add_edge(START, "dispatcher")

# dispatcher devuelve una lista de Send.
# lambda x: x significa: usa directamente lo que devolvio dispatcher como rutas dinamicas.
graph.add_conditional_edges("dispatcher", lambda x: x, ["specialist"])

# Todas las ramas specialist terminan en merge.
graph.add_edge("specialist", "merge")
graph.add_edge("merge", END)

app = graph.compile()
print("Grafo compilado.")

## Demo - Ejecutar consultas mixtas

Vamos a probar tres casos:

- Una consulta HR + Tech.
- Una consulta Billing + Tech.
- Una consulta fuera de alcance.

Lo importante no es solo la respuesta final. Miramos si aparecen varios bloques, por ejemplo `HR:` y `TECH:` en una misma salida.

In [ ]:
# Casos de prueba manual:
# 1. Consulta mixta HR + Tech.
# 2. Consulta mixta Billing + Tech.
# 3. Consulta fuera de alcance.
casos = [
    "tengo un problema con la VPN y necesito pedir vacaciones",
    "no puedo acceder al portal de reembolsos y perdi la contrasena",
    "que hay para cenar",
]

for q in casos:
    # results empieza como lista vacia porque las ramas van a acumular ahi.
    r = app.invoke({"query": q, "results": [], "final": ""})
    print(f"\nQ: {q}")
    print(r["final"])

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automaticos

Estos checks validan el comportamiento minimo esperado:

- Una consulta mixta debe activar mas de una rama.
- Una consulta simple de VPN debe activar Tech.

En Starter pueden fallar hasta completar los TODOs. En Resolution deben pasar.

In [ ]:
def run_checks():
    # Caso mixto: deberia disparar al menos Tech y HR.
    r1 = app.invoke({"query": "tengo problema con la VPN y necesito pedir dias libres", "results": [], "final": ""})
    assert "TECH" in r1["final"] and "HR" in r1["final"], f"deberia tener TECH y HR: {r1['final']}"

    # Caso simple: VPN deberia disparar Tech.
    r2 = app.invoke({"query": "problema con la VPN", "results": [], "final": ""})
    assert "TECH" in r2["final"], f"deberia tener TECH: {r2['final']}"

    print("Checks E09 OK")

run_checks()

## Que aprendiste hoy

- El routing simple elige una sola rama.
- El fan-out permite ejecutar varias ramas para una misma consulta.
- `Send` crea ramas dinamicas en runtime.
- `Annotated[list, operator.add]` evita que las ramas se pisen entre si.
- `merge` convierte resultados acumulados en una respuesta final.

Frase para clase:

> Cuando una consulta mezcla dominios, no queremos elegir un solo agente. Queremos delegar en paralelo y despues unir las respuestas.